# 28 — Advanced RAG: Retrieval Patterns for Network Knowledge

**Network LLM Engineer Certification — Advanced Retrieval**

### Learning goals
- Move beyond naive top-k vector retrieval
- Understand hybrid search, reciprocal rank fusion, multi-query retrieval, HyDE, parent-child retrieval and contextual compression
- Diagnose retrieval failures separately from generation failures

In [ ]:
%pip install -q sentence-transformers==5.7.0 scikit-learn>=1.5 pandas matplotlib networkx

In [ ]:
from pathlib import Path
def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path("/content/network_llm_engineer_certification")]:
        if (p / "CERTIFICATION_BLUEPRINT.md").exists():
            return p
    raise FileNotFoundError("Run from the extracted Network LLM Engineer Certification folder.")
ROOT = find_root()
DATA = ROOT / "data"
print("Course root:", ROOT)

## Why this belongs in a professional course

A basic RAG demo — chunk, embed, retrieve top-3 — is useful for learning, but production network knowledge has awkward characteristics:

- exact tokens matter (`BGP-3-NOTIFICATION`, `Eth1/49`, `0x80000001`);
- engineers phrase the same symptom differently;
- useful evidence may sit in a parent section while the exact match is in a small child chunk;
- a query can be underspecified;
- retrieved context can be redundant or contradictory.

The goal is to select a retrieval **strategy**, not worship one retriever.

## Pattern 1 — Hybrid retrieval

Combine:
- **sparse/lexical retrieval** for exact terms,
- **dense retrieval** for semantic meaning.

Then fuse rankings. A simple, robust fusion method is **Reciprocal Rank Fusion (RRF)**.

In [ ]:
import json, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

docs = [json.loads(x) for x in open(DATA/"mini_network_knowledge.jsonl", encoding="utf-8")]
texts = [d["text"] for d in docs]

tfidf = TfidfVectorizer(ngram_range=(1,2))
S = tfidf.fit_transform(texts)

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
D = embedder.encode(texts, normalize_embeddings=True)

def rank_sparse(q):
    qv = tfidf.transform([q])
    scores = (S @ qv.T).toarray().ravel()
    return list(np.argsort(-scores)), scores

def rank_dense(q):
    qv = embedder.encode([q], normalize_embeddings=True)[0]
    scores = D @ qv
    return list(np.argsort(-scores)), scores

def rrf(rankings, k=60):
    fused = {}
    for ranking in rankings:
        for rank, idx in enumerate(ranking, start=1):
            fused[idx] = fused.get(idx, 0) + 1/(k+rank)
    return sorted(fused, key=fused.get, reverse=True), fused

query = "small packets pass but large traffic fails through tunnel"
rs, _ = rank_sparse(query)
rd, _ = rank_dense(query)
rf, fused = rrf([rs, rd])

for idx in rf[:4]:
    print(docs[idx]["topic"], round(fused[idx], 4), "-", docs[idx]["text"][:110])

## Pattern 2 — Multi-query retrieval

One user question can be rewritten into several retrieval intents.

Example:

`"Remote EVPN host is unreachable"` →

- `"EVPN Type 2 MAC/IP route troubleshooting"`
- `"VXLAN VTEP underlay reachability"`
- `"VNI NVE mapping remote leaf"`
- `"EVPN control plane present data plane blackhole"`

Retrieve for each, then fuse/deduplicate.

In [ ]:
def multi_query_retrieve(queries, top_each=4):
    rankings = []
    for q in queries:
        ranking, _ = rank_dense(q)
        rankings.append(ranking[:top_each])
    fused, scores = rrf(rankings)
    return [(docs[i]["id"], docs[i]["topic"], scores[i]) for i in fused]

queries = [
    "EVPN MAC route troubleshooting",
    "VXLAN VTEP underlay reachability",
    "VNI mapping remote leaf",
]
print(multi_query_retrieve(queries)[:5])

## Pattern 3 — HyDE

**Hypothetical Document Embeddings** first generate a hypothetical answer/document, embed that, then retrieve real documents similar to it.

This can help when the query is short or vocabulary-poor. It can also amplify a bad hypothesis.

Network example:

Query: `peer won't come up`

Hypothetical answer might mention: `BGP neighbor, TCP/179, remote AS, source interface, reachability`.

That richer text can retrieve better documents — but only if the hypothesis generation is not wildly wrong.

In [ ]:
query = "peer won't come up"
hyde = (
    "A BGP peering session fails to establish. Verify neighbor address, configured remote AS, "
    "source address, IP reachability, TCP port 179, and BGP notification/log messages."
)
for label, q in [("raw", query), ("HyDE text", hyde)]:
    ranking, scores = rank_dense(q)
    print("\n", label)
    for i in ranking[:3]:
        print(round(float(scores[i]),3), docs[i]["topic"])

## Pattern 4 — Parent/child retrieval

Store small chunks for accurate matching but return the larger parent section for context.

Network documentation often has this shape:

- Parent: `BGP Session Establishment`
- Child: `Remote-AS mismatch`
- Child: `TCP/179 reachability`
- Child: `Update-source`
- Child: `OPEN notification`

The child finds the right concept; the parent gives enough surrounding procedure to answer safely.

## Pattern 5 — Contextual compression

Retrieve broadly, then remove passages that are irrelevant to the specific query before sending context to the generator.

This saves context-window budget and can reduce distraction. But compression is another model/system component that needs evaluation:
it can accidentally delete a critical caveat.

## Pattern 6 — Self-querying retrieval

When metadata is rich, translate a natural-language request into semantic text + metadata filters.

Example:

`"Show Junos EVPN incidents from the last 90 days"` →

- semantic query: `EVPN incident`
- vendor = `Junos`
- document_type = `incident`
- timestamp >= cutoff

Do not ask embeddings alone to solve structured filtering.

## Retrieval evaluation

Measure at least:

- **Recall@k:** did we retrieve at least one relevant item?
- **MRR:** how early was the first relevant result?
- **nDCG:** ranking quality when multiple graded-relevance results exist.
- **context precision:** how much retrieved context was actually useful?
- **grounded answer quality:** did generation stay supported?

A generator cannot recover a document that retrieval never supplied.

### Exercise — EVPN retrieval bake-off

Build 15–30 small documents containing:
- OSPF,
- BGP,
- EVPN/VXLAN,
- DNS,
- MTU,
- STP.

Create 10 queries with known relevant documents.

Compare:
1. sparse,
2. dense,
3. hybrid RRF,
4. multi-query + RRF.

Report Recall@3 and MRR. Then explain why the winner won.